# Porównanie Standardowego i Quasi-Hyperbolicznego Dyskontowania w MDPs

Ten notebook porównuje:
1. **Standard Q-Learning** - klasyczny algorytm z wykładniczym dyskontowaniem
2. **Quasi-Hyperbolic Q-Learning** - algorytm z present-bias (σ < 1)

## Główne różnice:
- Standard: $V(s) = E[\sum_{t=0}^\infty \gamma^t r_t]$
- QH: $V(s) = E[r_0 + \sigma \sum_{t=1}^\infty \gamma^t r_t]$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')

from experiments.comparison_standard_vs_qh import MDPComparison, StandardQLearning
from models.mdp_environments import InventoryMDP
from algorithms.qh_qlearning import QHQLearning

# Ustawienia wizualizacji
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 1. Konfiguracja środowiska i parametrów

In [ ]:
# Parametry środowiska
MAX_INVENTORY = 15
MAX_ORDER = 8

# Parametry uczenia
SIGMA = 0.7        # Present-bias parameter (σ < 1 oznacza preferencję dla teraźniejszości)
GAMMA = 0.95       # Discount factor
ALPHA = 0.1        # Learning rate
EPSILON = 0.1      # Exploration rate
N_EPISODES = 5000  # Liczba epizodów treningowych

print(f"Konfiguracja:")
print(f"  Sigma (present-bias): {SIGMA}")
print(f"  Gamma (discount): {GAMMA}")
print(f"  Liczba stanów: {MAX_INVENTORY + 1}")
print(f"  Liczba akcji: {MAX_ORDER + 1}")

## 2. Inicjalizacja środowiska i algorytmów

In [ ]:
# Tworzenie środowiska Inventory MDP
env = InventoryMDP(max_inventory=MAX_INVENTORY, max_order=MAX_ORDER)

# Inicjalizacja frameworku porównawczego
comparison = MDPComparison(
    env=env,
    sigma=SIGMA,
    gamma=GAMMA,
    alpha=ALPHA,
    epsilon=EPSILON
)

print(f"Środowisko utworzone: {env.n_states} stanów, {env.n_actions} akcji")

## 3. Trening obu algorytmów

In [ ]:
# Trenowanie obu algorytmów
comparison.train(n_episodes=N_EPISODES, record_interval=100)

print("\nTrening zakończony!")

## 4. Raport porównawczy

In [ ]:
# Generowanie i wyświetlanie raportu
report = comparison.generate_report()
print(report)

## 5. Wizualizacja porównawcza

In [ ]:
# Generowanie wykresów porównawczych
comparison.plot_comparison(save_path='../data/plots/comparison_results.png')

## 6. Szczegółowe porównanie polityk

In [ ]:
# Porównanie polityk
policy_comp = comparison.compare_policies()

print("Polityka Standard Q-Learning:")
print(policy_comp['standard_policy'])
print("\nPolityka QH Q-Learning:")
print(policy_comp['qh_policy'])
print(f"\nZgodność polityk: {policy_comp['agreement_percentage']:.1f}%")

if len(policy_comp['different_states']) > 0:
    print(f"\nStany gdzie polityki się różnią: {policy_comp['different_states']}")
    print("\nSzczegóły różnic:")
    for state in policy_comp['different_states']:
        print(f"  Stan {state}: Standard={policy_comp['standard_policy'][state]}, "
              f"QH={policy_comp['qh_policy'][state]}")

## 7. Porównanie funkcji wartości

In [ ]:
# Porównanie funkcji wartości
value_comp = comparison.compare_values()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Wykres funkcji wartości
ax = axes[0]
states = np.arange(env.n_states)
ax.plot(states, value_comp['standard_values'], 'o-', label='Standard', linewidth=2)
ax.plot(states, value_comp['qh_values'], 's-', label='QH', linewidth=2)
ax.set_xlabel('Stan (poziom zapasów)', fontsize=12)
ax.set_ylabel('Wartość V(s)', fontsize=12)
ax.set_title('Funkcje wartości', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Wykres różnic
ax = axes[1]
ax.bar(states, value_comp['value_difference'], 
       color=['red' if v < 0 else 'green' for v in value_comp['value_difference']],
       alpha=0.7)
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Stan (poziom zapasów)', fontsize=12)
ax.set_ylabel('Różnica wartości (Standard - QH)', fontsize=12)
ax.set_title(f"Różnice w wartościach\n(średnia: {value_comp['mean_abs_difference']:.3f})", fontsize=14)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../data/plots/value_function_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Średnia bezwzględna różnica: {value_comp['mean_abs_difference']:.4f}")
print(f"Maksymalna bezwzględna różnica: {value_comp['max_abs_difference']:.4f}")

## 8. Analiza spójności czasowej (Time-Consistency)

In [ ]:
# Analiza spójności czasowej
initial_state = env.n_states // 2  # Środkowy stan
consistency_analysis = comparison.analyze_time_consistency(initial_state=initial_state, horizon=15)

print(f"Analiza spójności czasowej dla stanu początkowego: {initial_state}")
print(f"Trajektoria: {consistency_analysis['trajectory']}")
print(f"Akcje: {consistency_analysis['actions']}")
print(f"\nCzy polityka jest czasowo spójna: {consistency_analysis['is_time_consistent']}")
print(f"Liczba niespójności: {len(consistency_analysis['inconsistencies'])}")

if not consistency_analysis['is_time_consistent']:
    print("\nSzczegóły niespójności czasowych:")
    for inc in consistency_analysis['inconsistencies']:
        print(f"  Krok {inc['time']}, Stan {inc['state']}: "
              f"Precommitted={inc['precommitted_action']}, "
              f"Myopic={inc['myopic_action']}")

## 9. Analiza wrażliwości na parametr σ

In [ ]:
# Analiza dla różnych wartości sigma
sigma_values = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
results = []

for sigma in sigma_values:
    print(f"\nTestowanie σ = {sigma}...")
    comp = MDPComparison(
        env=InventoryMDP(max_inventory=MAX_INVENTORY, max_order=MAX_ORDER),
        sigma=sigma,
        gamma=GAMMA,
        alpha=ALPHA,
        epsilon=EPSILON
    )
    comp.train(n_episodes=2000, record_interval=500)
    
    policy_comp = comp.compare_policies()
    value_comp = comp.compare_values()
    
    results.append({
        'sigma': sigma,
        'policy_agreement': policy_comp['agreement_percentage'],
        'mean_value_diff': value_comp['mean_abs_difference'],
        'max_value_diff': value_comp['max_abs_difference']
    })

# Wizualizacja wyników
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.plot([r['sigma'] for r in results], [r['policy_agreement'] for r in results], 
        'o-', linewidth=2, markersize=8)
ax.set_xlabel('σ (present-bias parameter)', fontsize=12)
ax.set_ylabel('Zgodność polityk (%)', fontsize=12)
ax.set_title('Wpływ σ na zgodność polityk', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 105])

ax = axes[1]
ax.plot([r['sigma'] for r in results], [r['mean_value_diff'] for r in results], 
        's-', linewidth=2, markersize=8, label='Średnia różnica')
ax.plot([r['sigma'] for r in results], [r['max_value_diff'] for r in results], 
        '^-', linewidth=2, markersize=8, label='Maksymalna różnica')
ax.set_xlabel('σ (present-bias parameter)', fontsize=12)
ax.set_ylabel('Różnica wartości', fontsize=12)
ax.set_title('Wpływ σ na różnice w wartościach', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/plots/sigma_sensitivity_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nPodsumowanie analizy wrażliwości:")
for r in results:
    print(f"σ={r['sigma']:.1f}: Zgodność={r['policy_agreement']:.1f}%, "
          f"Śr. różnica wartości={r['mean_value_diff']:.4f}")

## 10. Wnioski

### Kluczowe obserwacje:

1. **Różnice w politykach**: 
   - Im mniejsze σ (większy present-bias), tym większe różnice między politykami
   - Quasi-hyperbolic discounting prowadzi do bardziej "zachłannych" decyzji krótkoterminowych

2. **Funkcje wartości**:
   - Standard Q-learning zwykle wyżej wycenia przyszłe nagrody
   - QH Q-learning bardziej skupia się na nagrodach natychmiastowych

3. **Spójność czasowa**:
   - Standard Q-learning jest zawsze czasowo spójny
   - QH Q-learning może wykazywać niespójność czasową (precommitted vs myopic choices)

4. **Praktyczne implikacje**:
   - QH discounting lepiej modeluje zachowania ludzkie (present-bias)
   - Może prowadzić do suboptimalnych decyzji z perspektywy długoterminowej
   - Ważne dla zastosowań w ekonomii behawioralnej i finansach